# Notebook 8.1 - Training & Unity Catalog Registration

Train two small CNN models on MNIST, log both runs to MLflow, register them in Unity Catalog, and assign `champion` and `challenger` aliases.

In [ ]:
import os
import time
import numpy as np
import mlflow
import tensorflow as tf

from mlflow import MlflowClient
from mlflow.models import infer_signature

dbutils.widgets.text("catalog_name", "ai_ml_in_practice")
dbutils.widgets.text("schema_name", "mlops_workshop")
dbutils.widgets.text("registered_model_name", "mnist_cnn")
dbutils.widgets.text("experiment_path", f"/Users/{spark.sql('select current_user()').first()[0]}/8_mlops_mnist")
dbutils.widgets.text("train_limit", "12000")
dbutils.widgets.text("epochs", "2")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
registered_model_name = dbutils.widgets.get("registered_model_name")
experiment_path = dbutils.widgets.get("experiment_path")
train_limit = int(dbutils.widgets.get("train_limit"))
epochs = int(dbutils.widgets.get("epochs"))

full_model_name = f"{catalog_name}.{schema_name}.{registered_model_name}"

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

mlflow_client = MlflowClient()

print({
    "experiment_path": experiment_path,
    "full_model_name": full_model_name,
    "train_limit": train_limit,
    "epochs": epochs,
})

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train[:train_limit]
y_train = y_train[:train_limit]

validation_size = 2000
x_val = x_train[-validation_size:]
y_val = y_train[-validation_size:]
x_fit = x_train[:-validation_size]
y_fit = y_train[:-validation_size]

x_fit = np.expand_dims(x_fit, -1)
x_val = np.expand_dims(x_val, -1)
x_test_cnn = np.expand_dims(x_test, -1)

num_classes = 10
y_fit_cat = tf.keras.utils.to_categorical(y_fit, num_classes)
y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

print(x_fit.shape, x_val.shape, x_test_cnn.shape)

In [ ]:
def build_model_a():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])


def build_model_b():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(24, 5, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(48, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])


def train_and_register(model_label, build_fn):
    with mlflow.start_run(run_name=model_label) as run:
        model = build_fn()
        model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

        start = time.time()
        history = model.fit(
            x_fit,
            y_fit_cat,
            validation_data=(x_val, y_val_cat),
            epochs=epochs,
            batch_size=128,
            verbose=0,
        )
        train_seconds = time.time() - start

        test_loss, test_accuracy = model.evaluate(x_test_cnn, y_test_cat, verbose=0)
        predictions = model.predict(x_val[:50], verbose=0)
        signature = infer_signature(x_val[:50], predictions)

        mlflow.log_params({"model_label": model_label, "epochs": epochs, "train_limit": train_limit})
        mlflow.log_metrics({
            "val_accuracy_last": float(history.history["val_accuracy"][-1]),
            "test_accuracy": float(test_accuracy),
            "test_loss": float(test_loss),
            "train_seconds": float(train_seconds),
        })

        model_info = mlflow.tensorflow.log_model(
            model=model,
            artifact_path="model",
            signature=signature,
            registered_model_name=full_model_name,
        )

        latest_versions = mlflow_client.search_model_versions(f"name='{full_model_name}'")
        version = max(int(v.version) for v in latest_versions if v.run_id == run.info.run_id)
        return {
            "run_id": run.info.run_id,
            "version": version,
            "model_uri": model_info.model_uri,
            "test_accuracy": float(test_accuracy),
            "model_label": model_label,
        }


results = [
    train_and_register("model_a", build_model_a),
    train_and_register("model_b", build_model_b),
]

results = sorted(results, key=lambda row: row["test_accuracy"], reverse=True)
champion = results[0]
challenger = results[1]

results

In [ ]:
mlflow_client.set_registered_model_alias(name=full_model_name, alias="champion", version=str(champion["version"]))
mlflow_client.set_registered_model_alias(name=full_model_name, alias="challenger", version=str(challenger["version"]))

print({
    "champion_label": champion["model_label"],
    "champion_version": champion["version"],
    "champion_accuracy": champion["test_accuracy"],
    "challenger_label": challenger["model_label"],
    "challenger_version": challenger["version"],
    "challenger_accuracy": challenger["test_accuracy"],
})